This notebook collects heat hazard metrics for all census tracts in California, using the following indicators:

    - Heat days over 90F, historic and projected (mid-century, 2040-2070)
    - Heat days over 90F delta, projected - historic
    - 90F days (historic) are then normalized, 0-100
    - Air quality: combined normalized Ozone and PM2.5, historic

Heat and air quality are then multipled to create
    - heat_hazard_hist_idx per census tract

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd

## Import Data

In [14]:
ces = gpd.read_file("data_sources/hazards/CalEnviroScreen_4.0_Results.geojson")
len(ces)

8035

In [15]:
ces['tract'] = ces['tract'].astype(str)
ces['tract'] = ces['tract'].str.zfill(11)

In [16]:
ces_cols = ['tract', 'PollutionP', 'ozoneP', 'pmP', 'dieselP']
ces = ces[ces_cols]

In [17]:
fca = pd.read_csv("data_sources/hazards/heatdays_alltimes_tract.csv", dtype={"GEOID":object})
len(fca)

9109

In [18]:
fca_cols = [
    'GEOID',
    'days_over_80_historic',
    'days_over_80_midcentury',
    'days_over_90_historic',
    'days_over_90_midcentury',
    'days_over_100_historic',
    'days_over_100_midcentury',
]
fca = fca[fca_cols]

In [19]:
temp_thresholds = ["80", "90", "100"]
for temp in temp_thresholds:
    fca[f'delta_{temp}'] = fca[f'days_over_{temp}_midcentury'] - fca[f'days_over_{temp}_historic']

In [20]:
indices_data = pd.merge(fca, ces, left_on='GEOID', right_on='tract', how='left')
indices_data = indices_data.drop(columns=["tract"])

## Normalize

In [24]:
def normalize_indicators(df, columns):
    """
    Normalizes columns to a 0-1 scale using Min-Max scaling.
    Used for days/counts that aren't already percentiles.
    """
    for col in columns:
        col_min = df[col].min()
        col_max = df[col].max()
        if col_max != col_min:
            df[f'{col}_norm'] = ((df[col] - col_min) / (col_max - col_min)) * 100
        else:
            df[f'{col}_norm'] = 0
    return df

In [25]:
heat_cols = [
    'days_over_80_historic', 'days_over_80_midcentury',
    'days_over_90_historic', 'days_over_90_midcentury',
    'days_over_100_historic', 'days_over_100_midcentury',
]
indices_norm = normalize_indicators(indices_data, heat_cols)

## Create Indicies

In [32]:
indices_norm['AQI'] = indices_norm[['ozoneP', 'pmP', 'dieselP']].mean(axis=1)

In [33]:
indices_norm = normalize_indicators(indices_norm, ['AQI'])

In [39]:
indices_norm['heat_hazard_idx'] = indices_norm[['AQI_norm', 'days_over_90_historic_norm']].mean(axis=1)

In [40]:
idx = normalize_indicators(indices_norm, ['heat_hazard_idx'])

In [42]:
idx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9109 entries, 0 to 9108
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   GEOID                          9109 non-null   object 
 1   days_over_80_historic          9109 non-null   float64
 2   days_over_80_midcentury        9109 non-null   float64
 3   days_over_90_historic          9109 non-null   float64
 4   days_over_90_midcentury        9109 non-null   float64
 5   days_over_100_historic         9109 non-null   float64
 6   days_over_100_midcentury       9109 non-null   float64
 7   delta_80                       9109 non-null   float64
 8   delta_90                       9109 non-null   float64
 9   delta_100                      9109 non-null   float64
 10  PollutionP                     6858 non-null   float64
 11  ozoneP                         6858 non-null   float64
 12  pmP                            6858 non-null   f

## Export, keeping this limited for now

In [ ]:
cols_to_keep =
[
    'days_over_90_historic',
    'days_over_90_midcentury',
    'delta_90',
    'PollutionP',
    'AQI_norm',
    'heat_hazard_idx_norm'
]
